Indexing & Retrival

In [74]:
import os
from pathlib import Path
from dotenv import find_dotenv, load_dotenv, dotenv_values

print("Notebook working directory:", Path.cwd())

env_path = find_dotenv(".env", usecwd=True)
print("Found .env:", env_path or "NOT FOUND")

if env_path:
    print("Variables in .env:", list(dotenv_values(env_path)))
    print("Loaded:", load_dotenv(env_path, override=True))
    key_name = "XAI_API_KEY"  # Use the variable name from your tutorial
print("API key present:", bool(os.getenv(key_name)))

Notebook working directory: c:\Users\prsha\Desktop\DataAnalyst\Udemy_Python\0-DataIngestParsing\data
Found .env: c:\Users\prsha\Desktop\DataAnalyst\Udemy_Python\.env
Variables in .env: ['COURSE_ENV', 'XAI_API_KEY']
Loaded: True
API key present: True


Download Data

In [75]:
from pathlib import Path
from llama_index.readers.file import PDFReader
loader = PDFReader()
documents = loader.load_data(file=Path("transformers.pdf"))
len(documents)


15

In [76]:
documents[0].dict().keys()

dict_keys(['id_', 'embedding', 'metadata', 'excluded_embed_metadata_keys', 'excluded_llm_metadata_keys', 'relationships', 'metadata_template', 'metadata_separator', 'text_resource', 'image_resource', 'audio_resource', 'video_resource', 'text_template', 'class_name', 'text'])

In [77]:
documents[0].text


'Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗ †\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗ ‡\nillia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks that include an encoder and a decoder. The best\nperforming models also connect the encoder and decoder through an attention\nmechanism. We propose a new simple network architecture, the Transformer,\nbased solely on attention mechanisms, dispensing with recurren

In [78]:
documents[0].id_

'eead2d3d-222b-4d91-9ccf-6cdd56b26bd7'

Vector store Index

In [79]:
from llama_index.core import VectorStoreIndex
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

In [80]:
from llama_index.core import VectorStoreIndex
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_index = VectorStoreIndex.from_documents(
    documents,
    transformations=[SentenceSplitter(chunk_size=512, chunk_overlap=20)],
    embed_model=embed_model,
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8891.54it/s]


In [81]:
vector_index.index_struct.to_dict()

{'index_id': '29ecc165-2166-4a6b-8ab1-c9e148806a38',
 'summary': None,
 'nodes_dict': {'f47a670b-d0e2-4d31-ba4d-b737caa30364': 'f47a670b-d0e2-4d31-ba4d-b737caa30364',
  'b59b6a39-5323-4ea2-8a41-4f1eb8e70a59': 'b59b6a39-5323-4ea2-8a41-4f1eb8e70a59',
  'fd9d46d1-c53f-46fc-b126-1575255145b9': 'fd9d46d1-c53f-46fc-b126-1575255145b9',
  '47d91e36-12ec-494b-8a6a-cfc12c65d31f': '47d91e36-12ec-494b-8a6a-cfc12c65d31f',
  'eb55163e-ce8a-492e-9efe-c3982382dc3d': 'eb55163e-ce8a-492e-9efe-c3982382dc3d',
  'd208623a-f86b-495e-8745-7040b50c2be4': 'd208623a-f86b-495e-8745-7040b50c2be4',
  'bf7c7234-d4d4-4589-89b7-772c1449d82f': 'bf7c7234-d4d4-4589-89b7-772c1449d82f',
  'b892ef80-6ffb-4ea1-b53c-900a513842a5': 'b892ef80-6ffb-4ea1-b53c-900a513842a5',
  '76ce9f9b-0c4d-43ab-8801-a0d67b76a0dd': '76ce9f9b-0c4d-43ab-8801-a0d67b76a0dd',
  'f8387bdb-05f3-4a10-a3bf-a87efe355a46': 'f8387bdb-05f3-4a10-a3bf-a87efe355a46',
  '0f78f9a4-3940-4386-8f82-d32ec52ea146': '0f78f9a4-3940-4386-8f82-d32ec52ea146',
  'bd1dd8ac-6

vector Store Index as Retriever

In [82]:
vector_retriever = vector_index.as_retriever(similarity_top_k=3)   

In [83]:
node = vector_retriever.retrieve("What is a transformer model?")
len(node)

3

In [84]:
type(node)

list

In [85]:
node[0]


NodeWithScore(node=TextNode(id_='eb55163e-ce8a-492e-9efe-c3982382dc3d', embedding=None, metadata={'page_label': '3', 'file_name': 'transformers.pdf'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='5bd2895e-3ecf-4336-b2bf-8c9ca9f8f841', node_type='4', metadata={'page_label': '3', 'file_name': 'transformers.pdf'}, hash='b34a29fbeb23a13dbb728796ec20b7856060d6b4bd45d0ab2cdc9d4949a59a49')}, metadata_template='{key}: {value}', metadata_separator='\n', text='Figure 1: The Transformer - model architecture.\nThe Transformer follows this overall architecture using stacked self-attention and point-wise, fully\nconnected layers for both the encoder and decoder, shown in the left and right halves of Figure 1,\nrespectively.\n3.1 Encoder and Decoder Stacks\nEncoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two\nsub-layers. The first is a multi-head self-attention mechanism, a

Summary Index

In [88]:
from llama_index.core import SummaryIndex
summary_index = SummaryIndex.from_documents(documents)

In [89]:
summary_index.index_id
summary_index.index_struct.index_id 
summary_index.index_struct.to_dict()

{'index_id': '497b602b-30de-4f60-98fd-0c65faa7bea0',
 'summary': None,
 'nodes': ['6543cb90-b89d-48fd-8ab8-78e048d02936',
  'd128912b-0a4c-4547-a8ea-2d49751cb6c1',
  'ff0c3b45-276d-449a-8f8f-84550450a2ab',
  '4a209e1f-92bf-4e3f-aaf7-1a52719a0dce',
  '1a21e2de-6a5b-4236-9d92-74a0c29da0fa',
  'e5de1c88-a3c7-47ab-8cbe-e6990c8fba55',
  'd67c41a7-4a3d-4b1c-a657-020cbcece556',
  'e17451b8-65a2-4680-a3b5-47c3c6b474b5',
  '2b00acd8-e08b-45d2-9024-5df2124b2314',
  'e457448f-db9b-45ad-b615-ea7a6bd38d4a',
  'c2643c13-ca60-4379-b6fe-d665c898979b',
  '4d8c7577-518b-43cc-9465-7e8b69adbdbb',
  'dfb474c0-c052-45a7-9c93-99c3963be88d',
  'fcdd53a6-356e-4c26-9445-b4549e7f1900',
  '0da00579-df93-4c6b-ac57-0fa0f8bd6ca3']}

Summary Index LLM Retriever

In [90]:
summary_retriever = summary_index.as_retriever(retrieval_mode="llm")
nodes=summary_retriever.retrieve("What is  the use of positional encoding in transformer models? in 20 word")

In [94]:
for node in nodes:
  
    print("Node Id:", node.id_)
    print("Metadata:", node.metadata)
    print("end======================\n")

Node Id: 6543cb90-b89d-48fd-8ab8-78e048d02936
Metadata: {'page_label': '1', 'file_name': 'transformers.pdf'}
end======================

Node Id: d128912b-0a4c-4547-a8ea-2d49751cb6c1
Metadata: {'page_label': '2', 'file_name': 'transformers.pdf'}
end======================

Node Id: ff0c3b45-276d-449a-8f8f-84550450a2ab
Metadata: {'page_label': '3', 'file_name': 'transformers.pdf'}
end======================

Node Id: 4a209e1f-92bf-4e3f-aaf7-1a52719a0dce
Metadata: {'page_label': '4', 'file_name': 'transformers.pdf'}
end======================

Node Id: 1a21e2de-6a5b-4236-9d92-74a0c29da0fa
Metadata: {'page_label': '5', 'file_name': 'transformers.pdf'}
end======================

Node Id: e5de1c88-a3c7-47ab-8cbe-e6990c8fba55
Metadata: {'page_label': '6', 'file_name': 'transformers.pdf'}
end======================

Node Id: d67c41a7-4a3d-4b1c-a657-020cbcece556
Metadata: {'page_label': '7', 'file_name': 'transformers.pdf'}
end======================

Node Id: e17451b8-65a2-4680-a3b5-47c3c6b474b5
Me

Summary Index Embedding Retriever

In [97]:
summary_embed_retriever = summary_index.as_retriever(retrieval_mode="embedding",similarity_top_k=3)
nodes = summary_embed_retriever.retrieve("What is the use of positional encoding in transformer models?")

Keyword Table Index
Keyword Table Simple Retriever

In [100]:
import os
from dotenv import find_dotenv, load_dotenv
from llama_index.core import SimpleKeywordTableIndex
from llama_index.llms.openai_like import OpenAILike

env_path = find_dotenv(".env", usecwd=True)
if env_path:
    load_dotenv(env_path, override=True)

api_key = os.getenv("XAI_API_KEY", "").strip()
if not api_key:
    raise ValueError("XAI_API_KEY is missing. Add it to the project .env file.")

llm = OpenAILike(
    model="grok-3-mini",
    api_key=api_key,
    api_base="https://api.x.ai/v1",
    context_window=131072,
    is_chat_model=True,
    is_function_calling_model=False,
)
keyword_index = SimpleKeywordTableIndex.from_documents(
    documents,
    llm=llm,
    show_progress=True,
)

Extracting keywords from nodes: 100%|██████████| 15/15 [00:00<00:00, 219.20it/s]


Document Summary Index LLM Retriever

In [103]:
from llama_index.core import DocumentSummaryIndex

doc_summary_index = DocumentSummaryIndex.from_documents(
    documents,
    llm=llm,
    embed_model=embed_model,
    show_progress=True,
)
doc_summary_llm_retriever = doc_summary_index.as_retriever(retriever_mode="llm")
nodes = doc_summary_llm_retriever.retrieve("What is the use of positional encoding in transformer models?")

Summarizing documents:   0%|          | 0/15 [00:00<?, ?it/s]

current doc id: eead2d3d-222b-4d91-9ccf-6cdd56b26bd7


Summarizing documents:   0%|          | 0/15 [00:00<?, ?it/s]


BadRequestError: Error code: 400 - {'code': 'invalid-argument', 'error': 'Incorrect API key provided. You can obtain an API key from https://console.x.ai.'}